# 011 Decoding vs normalized component ratio with interval overlap checks

This notebook compares spatial AA and temporal AA as a function of normalized component ratio:

\[
K/D
\]

where spatial uses \(D=700\) and temporal uses \(D=300\).

It keeps the main decoding-accuracy plot, but adds configurable uncertainty intervals and asterisks above matched spatial/temporal points where the intervals do not overlap.

You can choose whether to display:

- SEM
- standard deviation
- approximate 95% confidence intervals
- custom multiplier intervals

The notebook also saves a matched statistics table.


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

FIT_SCOPE = "across"      # "within" or "across"
SUMMARY_KIND = "topm"     # "full" or "topm"
TOP_M_TO_PLOT = 10        # only used when SUMMARY_KIND = "topm"

DECODE_DIR_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}"

ANALYSIS_TYPES = ["spatial", "temporal"]
CONDITIONS = ["intact", "word", "rest"]

DIM_MAP = {
    "spatial": 700,
    "temporal": 300,
}

# Optional ratio window
RATIO_MIN = 0
RATIO_MAX = .15
# Example:
# RATIO_MIN = 0.10
# RATIO_MAX = 0.50

# Matching settings
MAX_RATIO_DIFF = 0.015
MATCH_METHOD = "symmetric_best"
# Options:
# "symmetric_best"       = best non-duplicated spatial/temporal pairs
# "temporal_to_spatial"  = each temporal K gets nearest spatial K
# "spatial_to_temporal"  = each spatial K gets nearest temporal K

# ============================================================
# UNCERTAINTY DISPLAY SETTINGS
# ============================================================

# Which uncertainty quantity to display on the main plot.
# Options: "sem", "std", "ci95", "custom"
ERROR_DISPLAY = "ci95"

# How to draw uncertainty.
# Options: "errorbar", "band", "none"
ERROR_STYLE = "errorbar"

# Used only if ERROR_DISPLAY = "custom"
CUSTOM_MULTIPLIER = 1.0

# Which interval to use to decide whether spatial and temporal are "different"
# by non-overlap.
# Options: "sem", "std", "ci95", "custom"
DIFFERENCE_INTERVAL = "ci95"

# Used only if DIFFERENCE_INTERVAL = "custom"
DIFFERENCE_CUSTOM_MULTIPLIER = 1.96

# If True, draw asterisks above points where intervals do not overlap.
SHOW_DIFFERENCE_ASTERISKS = True

# If True, use approximate z-test star as well. This is independent SEM approximation.
SHOW_ZTEST_ASTERISKS = False
ZTEST_ALPHA = 0.05

# Plot settings
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "011_decoding_normalized_component_ratio_stats_v8_clean"

SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

FIGSIZE = (9, 5.5)
YLIM = None

COND_COLORS = {
    "intact": "purple",
    "word": "green",
    "rest": "black",
}

ANALYSIS_LINESTYLES = {
    "spatial": "-",
    "temporal": ":",
}


ANALYSIS_MARKERS = {
    "spatial": None,
    "temporal": None,
}


# ============================================================
# COMPONENT LABEL SETTINGS
# ============================================================

SHOW_COMPONENT_LABELS = True

# Options:
#   "all"         = label all plotted K values
#   "significant" = label only interval-nonoverlap matched points
#   "none"        = label none
COMPONENT_LABEL_MODE = "significant"

# To reduce clutter, only label selected conditions.
# Use None to label all conditions.
LABEL_CONDITIONS = ["intact"]
# LABEL_CONDITIONS = None

# Label every nth point within each curve.
# 1 = label every point.
LABEL_EVERY_N = 1

# Label prefix options:
#   "K" gives K=50
#   "" gives 50
COMPONENT_LABEL_PREFIX = "K"

COMPONENT_LABEL_FONTSIZE = 7.5
COMPONENT_LABEL_ALPHA = 0.75


In [ ]:
# ============================================================
# IMPORTS AND HELPERS
# ============================================================

%matplotlib inline

from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

_fig_counter = 0

def save_current_fig(name):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    _fig_counter += 1
    safe = str(name).replace(" ", "_").replace("/", "-").replace("|", "_")
    safe = "".join(ch for ch in safe if ch.isalnum() or ch in ["_", "-", "."])
    out = FIG_DIR / f"{_fig_counter:03d}_{safe}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def add_two_part_legend(ax):
    """
    Clear publication legend:
    color = condition; line style = analysis type.
    """
    condition_handles = [
        Line2D([0], [0], color=COND_COLORS[c], lw=3, linestyle="-", label=c)
        for c in CONDITIONS
    ]

    analysis_handles = [
        Line2D([0], [0], color="black", lw=3, linestyle=ANALYSIS_LINESTYLES[a], label=a)
        for a in ANALYSIS_TYPES
    ]

    leg1 = ax.legend(
        handles=condition_handles,
        title="Condition",
        frameon=False,
        loc="upper left",
    )
    ax.add_artist(leg1)

    ax.legend(
        handles=analysis_handles,
        title="Analysis",
        frameon=False,
        loc="upper right",
    )

def should_label_condition(condition):
    if LABEL_CONDITIONS is None:
        return True
    return condition in LABEL_CONDITIONS

def make_component_label(K):
    if COMPONENT_LABEL_PREFIX:
        return f"{COMPONENT_LABEL_PREFIX}={int(K)}"
    return f"{int(K)}"

def annotate_component_labels_on_curve(ax, sub, analysis_type, condition, y_offset_frac=0.018):
    """
    Add K labels along a plotted curve.

    Uses a small vertical offset and labels every LABEL_EVERY_N points.
    """
    if not SHOW_COMPONENT_LABELS:
        return
    if COMPONENT_LABEL_MODE != "all":
        return
    if not should_label_condition(condition):
        return

    if len(sub) == 0:
        return

    ymin, ymax = ax.get_ylim()
    yrange = ymax - ymin

    sub = sub.sort_values("normalized_component_ratio").reset_index(drop=True)

    # slightly offset spatial and temporal labels in opposite directions
    direction = 1 if analysis_type == "temporal" else -1
    y_offset = direction * y_offset_frac * yrange

    for ii, row in sub.iterrows():
        if ii % LABEL_EVERY_N != 0:
            continue

        ax.text(
            row["normalized_component_ratio"],
            row["mean"] + y_offset,
            make_component_label(row["K"]),
            fontsize=COMPONENT_LABEL_FONTSIZE,
            alpha=COMPONENT_LABEL_ALPHA,
            ha="center",
            va="bottom" if direction > 0 else "top",
            color=COND_COLORS.get(condition, "black"),
        )

def annotate_significant_component_labels(ax, matched_stats):
    """
    Label only significant/non-overlapping matched points, and only on the winning curve.
    """
    if not SHOW_COMPONENT_LABELS:
        return
    if COMPONENT_LABEL_MODE != "significant":
        return
    if matched_stats is None or len(matched_stats) == 0:
        return

    ymin, ymax = ax.get_ylim()
    yrange = ymax - ymin

    label_conditions = LABEL_CONDITIONS if LABEL_CONDITIONS is not None else CONDITIONS

    sig = matched_stats[
        (matched_stats["interval_nonoverlap"]) &
        (matched_stats["condition"].isin(label_conditions))
    ].copy()

    for _, row in sig.iterrows():
        if row["mean_temporal"] > row["mean_spatial"]:
            x = row["ratio_temporal"]
            y = row["mean_temporal"] + row["display_halfwidth_temporal"] + 0.06 * yrange
            label = make_component_label(row["K_temporal"])
        else:
            x = row["ratio_spatial"]
            y = row["mean_spatial"] + row["display_halfwidth_spatial"] + 0.06 * yrange
            label = make_component_label(row["K_spatial"])

        ax.text(
            x,
            y,
            label,
            fontsize=COMPONENT_LABEL_FONTSIZE + 0.5,
            ha="center",
            va="bottom",
            color="black",
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.75),
        )

print("Figure directory:", FIG_DIR)


## Load decoding summaries

In [ ]:
# ============================================================
# LOAD DECODING SUMMARIES
# ============================================================

def standardize_df(df, analysis_type, fit_scope):
    df = df.copy()

    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "err" not in df.columns:
        rename["sem_accuracy"] = "err"
    if "std_accuracy" in df.columns and "std" not in df.columns:
        rename["std_accuracy"] = "std"
    if "sem" in df.columns and "err" not in df.columns:
        rename["sem"] = "err"
    df = df.rename(columns=rename)

    if "analysis_type" not in df.columns:
        df["analysis_type"] = analysis_type
    if "fit_scope" not in df.columns:
        df["fit_scope"] = fit_scope

    df["analysis_type"] = df["analysis_type"].astype(str)
    df["fit_scope"] = df["fit_scope"].astype(str)
    df["condition"] = df["condition"].astype(str)
    df["K"] = df["K"].astype(int)

    df = df[
        (df["analysis_type"] == analysis_type) &
        (df["fit_scope"] == fit_scope)
    ].copy()

    if SUMMARY_KIND == "topm":
        if "top_m" not in df.columns:
            raise ValueError("SUMMARY_KIND='topm', but no top_m column found.")
        df["top_m"] = df["top_m"].astype(int)
        df = df[df["top_m"] == TOP_M_TO_PLOT].copy()

    return df

def load_summary(analysis_type):
    d = Path(DECODE_DIR_TEMPLATE.format(
        analysis_type=analysis_type,
        fit_scope=FIT_SCOPE,
    ))

    fname = "full_summary.csv" if SUMMARY_KIND == "full" else "topm_summary.csv"
    path = d / fname

    if not path.exists():
        print("Missing:", path)
        return None

    print("Loaded:", path)
    return standardize_df(pd.read_csv(path), analysis_type, FIT_SCOPE)

dfs = []
for analysis_type in ANALYSIS_TYPES:
    df = load_summary(analysis_type)
    if df is not None and len(df):
        dfs.append(df)

decode_df = pd.concat(dfs, ignore_index=True)

decode_df["axis_dim"] = decode_df["analysis_type"].map(DIM_MAP)
decode_df["normalized_component_ratio"] = decode_df["K"] / decode_df["axis_dim"]

if RATIO_MIN is not None:
    decode_df = decode_df[decode_df["normalized_component_ratio"] >= RATIO_MIN].copy()
if RATIO_MAX is not None:
    decode_df = decode_df[decode_df["normalized_component_ratio"] <= RATIO_MAX].copy()

print("Rows:", len(decode_df))
display(decode_df.head())
display(decode_df.groupby(["analysis_type", "condition"])["K"].apply(list).reset_index())


## Define uncertainty intervals

In [ ]:
# ============================================================
# UNCERTAINTY HELPERS
# ============================================================

def infer_sem_and_std_columns(df):
    sem_col = None
    std_col = None

    for c in ["err", "sem", "stderr", "se", "sem_accuracy"]:
        if c in df.columns:
            sem_col = c
            break

    for c in ["std", "std_accuracy", "sd"]:
        if c in df.columns:
            std_col = c
            break

    return sem_col, std_col

SEM_COL, STD_COL = infer_sem_and_std_columns(decode_df)

print("SEM column:", SEM_COL)
print("STD column:", STD_COL)

if SEM_COL is None:
    raise ValueError("Could not find a SEM/error column. Expected err, sem, stderr, se, or sem_accuracy.")

def get_interval_values(df, interval_type, custom_multiplier=1.0):
    """
    Returns a vector of half-widths for plotting/overlap checks.
    """
    if interval_type == "sem":
        return df[SEM_COL].to_numpy()

    if interval_type == "ci95":
        return 1.96 * df[SEM_COL].to_numpy()

    if interval_type == "custom":
        return custom_multiplier * df[SEM_COL].to_numpy()

    if interval_type == "std":
        if STD_COL is None:
            raise ValueError("ERROR_DISPLAY='std' or DIFFERENCE_INTERVAL='std', but no std column found.")
        return df[STD_COL].to_numpy()

    raise ValueError("interval_type must be 'sem', 'std', 'ci95', or 'custom'.")

def interval_label(interval_type):
    if interval_type == "sem":
        return "SEM"
    if interval_type == "std":
        return "SD"
    if interval_type == "ci95":
        return "95% CI"
    if interval_type == "custom":
        return f"{CUSTOM_MULTIPLIER} × SEM"
    return interval_type

def nonoverlap(m1, h1, m2, h2):
    """
    True if intervals [m1-h1, m1+h1] and [m2-h2, m2+h2] do not overlap.
    """
    return (m1 + h1 < m2 - h2) or (m2 + h2 < m1 - h1)

print("Plotting interval:", interval_label(ERROR_DISPLAY))
print("Difference interval:", interval_label(DIFFERENCE_INTERVAL))


## Match spatial and temporal points by normalized ratio

In [ ]:
# ============================================================
# MATCH SPATIAL AND TEMPORAL POINTS
# ============================================================

def make_candidate_pairs_for_condition(df, condition):
    spatial = df[
        (df["analysis_type"] == "spatial") &
        (df["condition"] == condition)
    ].copy()

    temporal = df[
        (df["analysis_type"] == "temporal") &
        (df["condition"] == condition)
    ].copy()

    # Add row indices so we can recover interval values later.
    spatial = spatial.reset_index(drop=False).rename(columns={"index": "spatial_row"})
    temporal = temporal.reset_index(drop=False).rename(columns={"index": "temporal_row"})

    rows = []

    for _, s in spatial.iterrows():
        for _, t in temporal.iterrows():
            ratio_diff = abs(
                s["normalized_component_ratio"] -
                t["normalized_component_ratio"]
            )

            if ratio_diff <= MAX_RATIO_DIFF:
                rows.append({
                    "condition": condition,
                    "spatial_row": int(s["spatial_row"]),
                    "temporal_row": int(t["temporal_row"]),
                    "K_spatial": int(s["K"]),
                    "K_temporal": int(t["K"]),
                    "ratio_spatial": float(s["normalized_component_ratio"]),
                    "ratio_temporal": float(t["normalized_component_ratio"]),
                    "ratio_mid": float((s["normalized_component_ratio"] + t["normalized_component_ratio"]) / 2),
                    "ratio_diff": float(ratio_diff),
                    "mean_spatial": float(s["mean"]),
                    "mean_temporal": float(t["mean"]),
                })

    return pd.DataFrame(rows)

def select_pairs(candidates, method):
    if len(candidates) == 0:
        return candidates

    candidates = candidates.sort_values("ratio_diff").copy()

    if method == "temporal_to_spatial":
        return candidates.groupby(["condition", "K_temporal"], as_index=False).head(1).reset_index(drop=True)

    if method == "spatial_to_temporal":
        return candidates.groupby(["condition", "K_spatial"], as_index=False).head(1).reset_index(drop=True)

    if method == "symmetric_best":
        keep = []
        used_spatial = set()
        used_temporal = set()

        for _, row in candidates.iterrows():
            s_key = (row["condition"], int(row["K_spatial"]))
            t_key = (row["condition"], int(row["K_temporal"]))

            if s_key in used_spatial or t_key in used_temporal:
                continue

            keep.append(row)
            used_spatial.add(s_key)
            used_temporal.add(t_key)

        if len(keep) == 0:
            return pd.DataFrame(columns=candidates.columns)

        return pd.DataFrame(keep).reset_index(drop=True)

    raise ValueError("MATCH_METHOD must be symmetric_best, temporal_to_spatial, or spatial_to_temporal.")

candidate_pairs = pd.concat(
    [make_candidate_pairs_for_condition(decode_df, c) for c in CONDITIONS],
    ignore_index=True
)

matched_df = select_pairs(candidate_pairs, MATCH_METHOD)

# Safety fix: ensure ratio_mid exists even if a prior/older version failed.
if "ratio_mid" not in matched_df.columns and len(matched_df):
    matched_df["ratio_mid"] = (matched_df["ratio_spatial"] + matched_df["ratio_temporal"]) / 2

print("Candidate pairs:", len(candidate_pairs))
print("Matched pairs:", len(matched_df))
display(matched_df.head(30))


## Add interval-overlap statistics to matched points

In [ ]:
# ============================================================
# ADD INTERVAL / TEST STATISTICS
# ============================================================

# Prepare lookup tables for interval half-widths
plot_halfwidth = get_interval_values(
    decode_df,
    ERROR_DISPLAY,
    custom_multiplier=CUSTOM_MULTIPLIER,
)

diff_halfwidth = get_interval_values(
    decode_df,
    DIFFERENCE_INTERVAL,
    custom_multiplier=DIFFERENCE_CUSTOM_MULTIPLIER,
)

decode_df = decode_df.copy()
decode_df["_plot_halfwidth"] = plot_halfwidth
decode_df["_diff_halfwidth"] = diff_halfwidth

# Create row lookup
row_lookup = decode_df.reset_index().set_index("index")

stats_rows = []

for _, row in matched_df.iterrows():
    s = row_lookup.loc[int(row["spatial_row"])]
    t = row_lookup.loc[int(row["temporal_row"])]

    mean_spatial = float(s["mean"])
    mean_temporal = float(t["mean"])
    h_spatial = float(s["_diff_halfwidth"])
    h_temporal = float(t["_diff_halfwidth"])

    sem_spatial = float(s[SEM_COL])
    sem_temporal = float(t[SEM_COL])

    diff = mean_temporal - mean_spatial
    diff_se = np.sqrt(sem_spatial**2 + sem_temporal**2)
    z = diff / diff_se if diff_se > 0 else np.nan
    p = 2 * (1 - normal_cdf(abs(z))) if np.isfinite(z) else np.nan

    cur = row.to_dict()
    cur.update({
        "mean_spatial": mean_spatial,
        "mean_temporal": mean_temporal,
        "display_halfwidth_spatial": float(s["_plot_halfwidth"]),
        "display_halfwidth_temporal": float(t["_plot_halfwidth"]),
        "diff_halfwidth_spatial": h_spatial,
        "diff_halfwidth_temporal": h_temporal,
        "diff_temporal_minus_spatial": diff,
        "diff_se_independent": diff_se,
        "z_approx": z,
        "p_approx_two_sided": p,
        "interval_nonoverlap": nonoverlap(mean_spatial, h_spatial, mean_temporal, h_temporal),
        "approx_sig_p05": p < ZTEST_ALPHA if np.isfinite(p) else False,
        "winner": "temporal" if diff > 0 else ("spatial" if diff < 0 else "tie"),
    })

    stats_rows.append(cur)

matched_stats = pd.DataFrame(stats_rows)

if "ratio_mid" not in matched_stats.columns and len(matched_stats):
    matched_stats["ratio_mid"] = (matched_stats["ratio_spatial"] + matched_stats["ratio_temporal"]) / 2

display(
    matched_stats[
        [
            "condition",
            "K_spatial",
            "K_temporal",
            "ratio_spatial",
            "ratio_temporal",
            "ratio_mid",
            "ratio_diff",
            "mean_spatial",
            "mean_temporal",
            "diff_temporal_minus_spatial",
            "interval_nonoverlap",
            "approx_sig_p05",
            "winner",
        ]
    ].sort_values(["condition", "ratio_mid"]).head(80)
)

print("Non-overlap counts:")
display(
    matched_stats.groupby(["condition", "winner"])[
        ["interval_nonoverlap", "approx_sig_p05"]
    ].sum().reset_index()
)


## Main plot: curves with configurable intervals and asterisks for non-overlap

In [ ]:
# ============================================================
# MAIN PLOT: CURVES + INTERVALS + ASTERISKS
# ============================================================

suffix = "full" if SUMMARY_KIND == "full" else f"top-{TOP_M_TO_PLOT}"
error_label = interval_label(ERROR_DISPLAY)
difference_label = interval_label(DIFFERENCE_INTERVAL)

fig, ax = plt.subplots(figsize=FIGSIZE)

# Plot curves
for analysis_type in ANALYSIS_TYPES:
    for condition in CONDITIONS:
        sub = decode_df[
            (decode_df["analysis_type"] == analysis_type) &
            (decode_df["condition"] == condition)
        ].sort_values("normalized_component_ratio")

        if len(sub) == 0:
            continue

        x = sub["normalized_component_ratio"].to_numpy()
        y = sub["mean"].to_numpy()
        h = sub["_plot_halfwidth"].to_numpy()

        if ERROR_STYLE == "errorbar":
            ax.errorbar(
                x,
                y,
                yerr=h,
                                capsize=4,
                linewidth=2.7,
                color=COND_COLORS[condition],
                linestyle=ANALYSIS_LINESTYLES[analysis_type],
                label=f"{analysis_type} | {condition}",
            )
        else:
            ax.plot(
                x,
                y,
                                linewidth=2.7,
                color=COND_COLORS[condition],
                linestyle=ANALYSIS_LINESTYLES[analysis_type],
                label=f"{analysis_type} | {condition}",
            )

            if ERROR_STYLE == "band":
                ax.fill_between(
                    x,
                    y - h,
                    y + h,
                    color=COND_COLORS[condition],
                    alpha=0.10,
                )

# Add asterisks where matched intervals do not overlap
if SHOW_DIFFERENCE_ASTERISKS and len(matched_stats):
    ymin, ymax = plt.ylim()
    yrange = ymax - ymin

    for condition in CONDITIONS:
        sub = matched_stats[
            (matched_stats["condition"] == condition) &
            (matched_stats["interval_nonoverlap"])
        ].copy()

        if len(sub) == 0:
            continue

        for _, row in sub.iterrows():
            x = row["ratio_mid"]

            # Place star just above the higher of the two intervals
            top_spatial = row["mean_spatial"] + row["display_halfwidth_spatial"]
            top_temporal = row["mean_temporal"] + row["display_halfwidth_temporal"]
            y = max(top_spatial, top_temporal) + 0.03 * yrange

            ax.text(
                x,
                y,
                "*",
                fontsize=18,
                ha="center",
                va="bottom",
                color=COND_COLORS[condition],
                fontweight="bold",
            )

# Optional approximate z-test stars, plotted as smaller plus signs
if SHOW_ZTEST_ASTERISKS and len(matched_stats):
    ymin, ymax = plt.ylim()
    yrange = ymax - ymin

    for condition in CONDITIONS:
        sub = matched_stats[
            (matched_stats["condition"] == condition) &
            (matched_stats["approx_sig_p05"])
        ].copy()

        for _, row in sub.iterrows():
            x = row["ratio_mid"]
            top_spatial = row["mean_spatial"] + row["display_halfwidth_spatial"]
            top_temporal = row["mean_temporal"] + row["display_halfwidth_temporal"]
            y = max(top_spatial, top_temporal) + 0.08 * yrange

            ax.text(
                x,
                y,
                "+",
                fontsize=14,
                ha="center",
                va="bottom",
                color=COND_COLORS[condition],
                fontweight="bold",
            )

if YLIM is not None:
    ax.set_ylim(*YLIM)

ax.set_xlabel("Normalized component ratio (K / axis dimensionality)")
ax.set_ylabel("Decoding accuracy")
ax.set_title(
    f"Decoding vs normalized ratio | {FIT_SCOPE} | {suffix}\n"
    f"display={error_label}; stars={difference_label} non-overlap"
)
add_two_part_legend(ax)
plt.tight_layout()

save_current_fig(
    f"curves_with_{ERROR_DISPLAY}_and_{DIFFERENCE_INTERVAL}_nonoverlap_stars_{FIT_SCOPE}_{suffix}"
)
plt.show()
plt.close()


## Optional: compact matched-points-only plot

In [ ]:
# ============================================================
# OPTIONAL PLOT: MATCHED POINTS ONLY
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE)

for condition in CONDITIONS:
    sub = matched_stats[matched_stats["condition"] == condition].sort_values("ratio_mid")

    if len(sub) == 0:
        continue

    # Spatial matched points
    ax.errorbar(
        sub["ratio_mid"],
        sub["mean_spatial"],
        yerr=sub["display_halfwidth_spatial"],
        fmt="o-",
        capsize=4,
        linewidth=2.5,
        color=COND_COLORS[condition],
        linestyle="-",
        label=f"spatial | {condition}",
    )

    # Temporal matched points
    ax.errorbar(
        sub["ratio_mid"],
        sub["mean_temporal"],
        yerr=sub["display_halfwidth_temporal"],
        fmt="o--",
        capsize=4,
        linewidth=2.5,
        color=COND_COLORS[condition],
        linestyle="--",
        label=f"temporal | {condition}",
    )

    sig = sub[sub["interval_nonoverlap"]]
    for _, row in sig.iterrows():
        y = max(
            row["mean_spatial"] + row["display_halfwidth_spatial"],
            row["mean_temporal"] + row["display_halfwidth_temporal"],
        )
        ax.text(
            row["ratio_mid"],
            y,
            "*",
            fontsize=18,
            ha="center",
            va="bottom",
            color=COND_COLORS[condition],
            fontweight="bold",
        )


annotate_significant_component_labels(ax, matched_stats)

if YLIM is not None:
    ax.set_ylim(*YLIM)

ax.set_xlabel("Matched normalized component ratio")
ax.set_ylabel("Decoding accuracy")
ax.set_title(
    f"Matched spatial-temporal points | {FIT_SCOPE} | {suffix}\n"
    f"display={error_label}; stars={difference_label} non-overlap"
)
add_two_part_legend(ax)
plt.tight_layout()

save_current_fig(
    f"matched_points_with_{ERROR_DISPLAY}_and_{DIFFERENCE_INTERVAL}_stars_{FIT_SCOPE}_{suffix}"
)
plt.show()
plt.close()


## Save matched statistics table

In [ ]:
# ============================================================
# SAVE STATS TABLE
# ============================================================

out_csv = FIG_DIR / f"matched_spatial_temporal_interval_stats_{FIT_SCOPE}_{suffix}.csv"
matched_stats.to_csv(out_csv, index=False)
print("Saved:", out_csv)

display(
    matched_stats[
        [
            "condition",
            "K_spatial",
            "K_temporal",
            "ratio_spatial",
            "ratio_temporal",
            "ratio_mid",
            "ratio_diff",
            "mean_spatial",
            "display_halfwidth_spatial",
            "mean_temporal",
            "display_halfwidth_temporal",
            "diff_temporal_minus_spatial",
            "interval_nonoverlap",
            "approx_sig_p05",
            "winner",
        ]
    ].sort_values(["condition", "ratio_mid"]).head(120)
)
